# MINE4201 - SR - Taller 1 - Experimentación y optimización para K vecinos más cercanos

**Grupo 3**

Lina María Ojeda Amaya Código: 202112324

Diego Felipe Tolosa Código: 201413235

Juan Manuel Rivera López Código: 201534131

Johana Alejandra Rátiva Mora Código: 202513844

# Importación de librerías y datos

In [145]:
import numpy as np
import os
import pandas as pd
import random
from matplotlib import pyplot as plt
import time

from scipy.sparse import csr_matrix

from sklearn.preprocessing import normalize
from tqdm import tqdm

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)

In [116]:
base_path = "data/"
links_path = os.path.join(base_path, "link.csv")
movies_path = os.path.join(base_path, "movie.csv")
ratings_path = os.path.join(base_path, "rating.csv")

In [117]:
ratings = pd.read_csv(ratings_path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [118]:
movies = pd.read_csv(movies_path)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [119]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings.drop(test.index)

In [120]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

In [121]:
user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))

# Evaluación de similitud usuario-usuario

In [122]:
N = R_train.shape[0]

counts = np.diff(R_train.indptr)
sums = np.add.reduceat(R_train.data, R_train.indptr[:-1])
means = sums / counts

### Función de cálculo de rating - User-User

Se crea una función capaz de implementar la predicción de ratings mediante **colaboración basada en usuarios (user-user collaborative filtering)**. Esta función calcula la predicción del rating de un usuario para un ítem específico utilizando los ratings de usuarios similares.

#### `predict_rating_user()`

La función toma como entrada los parámetros necesarios para realizar la predicción:

- **R**: Matriz dispersa (usuario × ítem) que contiene los ratings de entrenamiento
- **userId**: Índice del usuario para el cual se realiza la predicción
- **movieId**: Índice del ítem a predecir
- **neighbours**: Array con los índices de los usuarios más similares
- **sims**: Array con las similitudes correspondientes a cada vecino
- **K**: Número máximo de vecinos considerados
- **sim_threshold**: Valor mínimo de similitud para incluir un vecino en la predicción
- **means**: Promedio de ratings para cada usuario
- **weighting**: Booleano para aplicar ponderación de McLaughlin (opcional)
- **B**: Matriz de co-rated (elementos en común precomputados) (requerida si weighting=True)
- **beta**: Parámetro de escala para la ponderación de McLaughlin (por defecto 50)

#### Proceso de cálculo

1. **Extracción de ratings**: Se obtienen los ratings que los usuarios similares han dado al ítem objetivo
2. **Filtrado**: Se descartan vecinos que no calificaron el ítem o cuya similitud es inferior al threshold
3. **Centrado**: Se restan los promedios de cada usuario para normalizar las escalas
4. **Ponderación (opcional)**: Se aplica el método de McLaughlin para penalizar similitudes basadas en pocos datos comunes
5. **Combinación ponderada**: Se calcula la predicción como el promedio del usuario más una desviación ponderada

#### Fórmula matemática

$$\hat{r}_{u,i} = \bar{r}_u + \frac{\sum_{v \in N_i(u)} w_v \cdot \text{sim}(u,v) \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N_i(u)} |w_v \cdot \text{sim}(u,v)|}$$

Donde $w_v = \min(\text{corated}(u,v) / \beta, 1.0)$ cuando se aplica ponderación de McLaughlin.

Esta implementación permite optimizar las predicciones al penalizar similitudes con poca base empírica, mejorando la robustez del modelo.

In [146]:
def predict_rating_user(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    # --- get ratings for the movie ---
    ratings = R[neighbours, movieId].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = B[userId]

        weights = np.minimum(intersections / beta, 1.0)

        weights = weights[mask]

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[userId] + numerator / denominator

    return pred

### Función de evaluación RMSE - User-User

Esta función calcula el error cuadrático medio (RMSE) de las predicciones generadas por el modelo de filtrado colaborativo basado en usuarios. Permite comparar el desempeño del sistema bajo diferentes configuraciones de vecinos, similitud y ponderación.

#### `evaluate_rmse_user()`

**Parámetros:**

- **R**: Matriz dispersa de ratings de entrenamiento

- **test_df**: DataFrame de prueba con índices de usuario e ítem

- **neighbors_list**: Matriz de vecinos para cada usuario

- **sims_list**: Matriz de similitudes para cada usuario

- **K**: Número de vecinos considerados

- **sim_threshold**: Umbral mínimo de similitud

- **means**: Promedio de ratings por usuario

- **weighting**: Si aplica ponderación de McLaughlin

- **B**: Matriz de co-rated (elementos en común precomputados)

- **beta**: Parámetro de escala para ponderación


#### Proceso de cálculo

1. Para cada usuario y película en el set de prueba, predice el rating usando la función de predicción.
2. Guarda la predicción y el rating real.
3. Calcula el RMSE entre las predicciones y los ratings reales.

#### Fórmula matemática

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (\hat{r}_i - r_i)^2}$$

Donde $\hat{r}_i$ es la predicción y $r_i$ el rating real.

Esta función permite evaluar la calidad de las predicciones y comparar configuraciones del modelo.

In [150]:
def evaluate_rmse_user(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    preds = []
    truths = []

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating_user(
            R,
            user_idx,
            movie_idx,
            neighbors_list[user_idx],
            sims_list[user_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B,
            beta=beta
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse

## Sub muestra de test (n=5.000)

In [151]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):
            weight=False
            rmse = evaluate_rmse_user(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weight,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse))
            print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, False, np.float64(0.8754917505297362))


('cosine', 25, 0.3, False, np.float64(0.8720686704904383))


('cosine', 25, 0.5, False, np.float64(0.8994676945490273))


...

('jaccard', 50, 0.5, False, np.float64(0.9168448586680615))


('jaccard', 100, 0, False, np.float64(0.8537566672461587))


('jaccard', 100, 0.3, False, np.float64(0.8855272251516845))




Similarity metric: 100%|██████████| 3/3 [00:13<00:00,  4.66s/it]

('jaccard', 100, 0.5, False, np.float64(0.9213708555556887))


In [152]:
results

[('cosine', 25, 0, False, np.float64(0.8754917505297362)),
 ('cosine', 25, 0.3, False, np.float64(0.8720686704904383)),
 ('cosine', 25, 0.5, False, np.float64(0.8994676945490273)),
 ('cosine', 50, 0, False, np.float64(0.8523012503228641)),
 ('cosine', 50, 0.3, False, np.float64(0.8491847943040124)),
 ('cosine', 50, 0.5, False, np.float64(0.8839743801272706)),
 ('cosine', 100, 0, False, np.float64(0.8449797573858153)),
 ('cosine', 100, 0.3, False, np.float64(0.8452799129614096)),
 ('cosine', 100, 0.5, False, np.float64(0.8797395197854984)),
 ('pearson', 25, 0, False, np.float64(0.8691050174067901)),
 ('pearson', 25, 0.3, False, np.float64(0.8438792388104881)),
 ('pearson', 25, 0.5, False, np.float64(0.8610410424811081)),
 ('pearson', 50, 0, False, np.float64(0.8562168766214626)),
 ('pearson', 50, 0.3, False, np.float64(0.8499879946737761)),
 ('pearson', 50, 0.5, False, np.float64(0.8437798831131719)),
 ('pearson', 100, 0, False, np.float64(0.845019019260748)),
 ('pearson', 100, 0.3, Fal

Viendo que el coeficiente de Jaccard tiene los peores resultados en la exploración incial, se excluirá en iteraciones futuras.

## Submuestra de test (n=250.000)

In [153]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

for metric in tqdm(["cosine", "pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):

            rmse = evaluate_rmse_user(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse))
            print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/2 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, False, np.float64(0.8806656033893725))


('cosine', 25, 0.3, False, np.float64(0.8741087211940434))


('cosine', 25, 0.5, False, np.float64(0.8753475981812547))


...

('pearson', 50, 0.5, False, np.float64(0.8177949505761316))


('pearson', 100, 0, False, np.float64(0.8581761219436683))


('pearson', 100, 0.3, False, np.float64(0.8513725760154347))




Similarity metric: 100%|██████████| 2/2 [07:32<00:00, 226.32s/it]

('pearson', 100, 0.5, False, np.float64(0.8170917909421229))


In [154]:
results

[('cosine', 25, 0, False, np.float64(0.8806656033893725)),
 ('cosine', 25, 0.3, False, np.float64(0.8741087211940434)),
 ('cosine', 25, 0.5, False, np.float64(0.8753475981812547)),
 ('cosine', 50, 0, False, np.float64(0.8626560533342876)),
 ('cosine', 50, 0.3, False, np.float64(0.8573224019235202)),
 ('cosine', 50, 0.5, False, np.float64(0.8684518666093048)),
 ('cosine', 100, 0, False, np.float64(0.8511609839977688)),
 ('cosine', 100, 0.3, False, np.float64(0.8477897557746596)),
 ('cosine', 100, 0.5, False, np.float64(0.864028738041872)),
 ('pearson', 25, 0, False, np.float64(0.8892202755242916)),
 ('pearson', 25, 0.3, False, np.float64(0.8634113679362047)),
 ('pearson', 25, 0.5, False, np.float64(0.817639193129947)),
 ('pearson', 50, 0, False, np.float64(0.8733750963456223)),
 ('pearson', 50, 0.3, False, np.float64(0.8562543751332603)),
 ('pearson', 50, 0.5, False, np.float64(0.8177949505761316)),
 ('pearson', 100, 0, False, np.float64(0.8581761219436683)),
 ('pearson', 100, 0.3, Fals

## Implementación de Ponderación de McLaughlin

Se calculará McNally para el mejor tipo de similitud en el modelo usuario-usuario, que en este caso es Pearson, para esta se iterará para 3 Valores de k vecinos, 3 valores de umbral de similitud, y 5 posibles valores de ponderación de McLaughlin (incluyendo NO implementarla)

## Submuestra de test (n=5.000)

In [155]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(5_000, random_state=42)

for metric in tqdm(["pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/users_{metric}_top100_co_rated.npz") as data:
        top_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.3, 0.5], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse = evaluate_rmse_user(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse))
                    print((metric, K, sim_threshold, weight, None, rmse))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse = evaluate_rmse_user(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse))
                        print((metric, K, sim_threshold, weight, beta, rmse))

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: -1.0
max sim: 0.9672295
min co_rated: 0
max co_rated: 3642


('pearson', 25, 0, False, None, np.float64(0.8691050174067901))
('pearson', 25, 0, True, 5, np.float64(0.869541483445122))
('pearson', 25, 0, True, 10, np.float64(0.8702049356821959))
('pearson', 25, 0, True, 25, np.float64(0.87163840420329))
('pearson', 25, 0, True, 50, np.float64(0.8734886457752691))


('pearson', 25, 0, True, 100, np.float64(0.8740969481708274))
('pearson', 25, 0.3, False, None, np.float64(0.8438792388104881))
('pearson', 25, 0.3, True, 5, np.float64(0.8442698949274675))
('pearson', 25, 0.3, True, 10, np.float64(0.84565113913288))
('pearson', 25, 0.3, True, 25, np.float64(0.8454252121054788))
('pearson', 25, 0.3, True, 50, np.float64(0.845324726764283))


('pearson', 25, 0.3, True, 100, np.float64(0.845302041970162))
('pearson', 25, 0.5, False, None, np.float64(0.8610410424811081))
('pearson', 25, 0.5, True, 5, np.float64(0.8609702495777507))
('pearson', 25, 0.5, True, 10, np.float64(0.8614994606480959))
('pearson', 25, 0.5, True, 25, np.float64(0.8604545955264193))
('pearson', 25, 0.5, True, 50, np.float64(0.8603277299850572))


...

('pearson', 100, 0, False, None, np.float64(0.845019019260748))
('pearson', 100, 0, True, 5, np.float64(0.8454372405738101))
('pearson', 100, 0, True, 10, np.float64(0.8461970071268775))
('pearson', 100, 0, True, 25, np.float64(0.845326369799388))
('pearson', 100, 0, True, 50, np.float64(0.8453878527769222))


('pearson', 100, 0, True, 100, np.float64(0.8456585230023135))
('pearson', 100, 0.3, False, None, np.float64(0.8564960285905605))
('pearson', 100, 0.3, True, 5, np.float64(0.8577149926268149))
('pearson', 100, 0.3, True, 10, np.float64(0.8589418076497167))
('pearson', 100, 0.3, True, 25, np.float64(0.8587211743648868))
('pearson', 100, 0.3, True, 50, np.float64(0.8590352205831374))


('pearson', 100, 0.3, True, 100, np.float64(0.859290705817382))
('pearson', 100, 0.5, False, None, np.float64(0.8433968379717605))
('pearson', 100, 0.5, True, 5, np.float64(0.8433245639264784))
('pearson', 100, 0.5, True, 10, np.float64(0.8438561833988775))
('pearson', 100, 0.5, True, 25, np.float64(0.8424163646200066))
('pearson', 100, 0.5, True, 50, np.float64(0.842307965457774))




Similarity metric: 100%|██████████| 1/1 [00:24<00:00, 24.49s/it]

('pearson', 100, 0.5, True, 100, np.float64(0.842307965457774))


In [156]:
results

[('pearson', 25, 0, False, None, np.float64(0.8691050174067901)),
 ('pearson', 25, 0, True, 5, np.float64(0.869541483445122)),
 ('pearson', 25, 0, True, 10, np.float64(0.8702049356821959)),
 ('pearson', 25, 0, True, 25, np.float64(0.87163840420329)),
 ('pearson', 25, 0, True, 50, np.float64(0.8734886457752691)),
 ('pearson', 25, 0, True, 100, np.float64(0.8740969481708274)),
 ('pearson', 25, 0.3, False, None, np.float64(0.8438792388104881)),
 ('pearson', 25, 0.3, True, 5, np.float64(0.8442698949274675)),
 ('pearson', 25, 0.3, True, 10, np.float64(0.84565113913288)),
 ('pearson', 25, 0.3, True, 25, np.float64(0.8454252121054788)),
 ('pearson', 25, 0.3, True, 50, np.float64(0.845324726764283)),
 ('pearson', 25, 0.3, True, 100, np.float64(0.845302041970162)),
 ('pearson', 25, 0.5, False, None, np.float64(0.8610410424811081)),
 ('pearson', 25, 0.5, True, 5, np.float64(0.8609702495777507)),
 ('pearson', 25, 0.5, True, 10, np.float64(0.8614994606480959)),
 ('pearson', 25, 0.5, True, 25, np.f

## Submuestra de test (n=250.000)

In [157]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

for metric in tqdm(["pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/users_{metric}_top100_co_rated.npz") as data:
        top_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.3, 0.5], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse = evaluate_rmse_user(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse))
                    print((metric, K, sim_threshold, weight, None, rmse))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse = evaluate_rmse_user(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse))
                        print((metric, K, sim_threshold, weight, beta, rmse))


Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: -1.0
max sim: 0.9672295
min co_rated: 0
max co_rated: 3642


('pearson', 25, 0, False, None, np.float64(0.8892202755242916))
('pearson', 25, 0, True, 5, np.float64(0.8894383992489105))
('pearson', 25, 0, True, 10, np.float64(0.8897755943753323))
('pearson', 25, 0, True, 25, np.float64(0.8904079461075728))
('pearson', 25, 0, True, 50, np.float64(0.8910818096581642))


('pearson', 25, 0, True, 100, np.float64(0.8919589438108231))
('pearson', 25, 0.3, False, None, np.float64(0.8634113679362047))
('pearson', 25, 0.3, True, 5, np.float64(0.8636374015578946))
('pearson', 25, 0.3, True, 10, np.float64(0.863678859668834))
('pearson', 25, 0.3, True, 25, np.float64(0.8637344917114653))
('pearson', 25, 0.3, True, 50, np.float64(0.8636260056190213))


('pearson', 25, 0.3, True, 100, np.float64(0.8637106172409938))
('pearson', 25, 0.5, False, None, np.float64(0.817639193129947))
('pearson', 25, 0.5, True, 5, np.float64(0.8176636380750716))
('pearson', 25, 0.5, True, 10, np.float64(0.817881875788543))
('pearson', 25, 0.5, True, 25, np.float64(0.8179769340579557))
('pearson', 25, 0.5, True, 50, np.float64(0.8180141009139629))


...

('pearson', 100, 0, False, None, np.float64(0.8581761219436683))
('pearson', 100, 0, True, 5, np.float64(0.8583942314060745))
('pearson', 100, 0, True, 10, np.float64(0.8585982513536641))
('pearson', 100, 0, True, 25, np.float64(0.8589647409506121))
('pearson', 100, 0, True, 50, np.float64(0.8595523837997658))


('pearson', 100, 0, True, 100, np.float64(0.8602099614333594))
('pearson', 100, 0.3, False, None, np.float64(0.8513725760154347))
('pearson', 100, 0.3, True, 5, np.float64(0.851481214747705))
('pearson', 100, 0.3, True, 10, np.float64(0.8513770388524527))
('pearson', 100, 0.3, True, 25, np.float64(0.8510990026022691))
('pearson', 100, 0.3, True, 50, np.float64(0.8509024069990727))


('pearson', 100, 0.3, True, 100, np.float64(0.8509659255187818))
('pearson', 100, 0.5, False, None, np.float64(0.8170917909421229))
('pearson', 100, 0.5, True, 5, np.float64(0.8171143736817182))
('pearson', 100, 0.5, True, 10, np.float64(0.8172678670035585))
('pearson', 100, 0.5, True, 25, np.float64(0.8171070738539394))
('pearson', 100, 0.5, True, 50, np.float64(0.8171364467630272))




Similarity metric: 100%|██████████| 1/1 [20:11<00:00, 1211.07s/it]

('pearson', 100, 0.5, True, 100, np.float64(0.8171453153467022))


In [158]:
results

[('pearson', 25, 0, False, None, np.float64(0.8892202755242916)),
 ('pearson', 25, 0, True, 5, np.float64(0.8894383992489105)),
 ('pearson', 25, 0, True, 10, np.float64(0.8897755943753323)),
 ('pearson', 25, 0, True, 25, np.float64(0.8904079461075728)),
 ('pearson', 25, 0, True, 50, np.float64(0.8910818096581642)),
 ('pearson', 25, 0, True, 100, np.float64(0.8919589438108231)),
 ('pearson', 25, 0.3, False, None, np.float64(0.8634113679362047)),
 ('pearson', 25, 0.3, True, 5, np.float64(0.8636374015578946)),
 ('pearson', 25, 0.3, True, 10, np.float64(0.863678859668834)),
 ('pearson', 25, 0.3, True, 25, np.float64(0.8637344917114653)),
 ('pearson', 25, 0.3, True, 50, np.float64(0.8636260056190213)),
 ('pearson', 25, 0.3, True, 100, np.float64(0.8637106172409938)),
 ('pearson', 25, 0.5, False, None, np.float64(0.817639193129947)),
 ('pearson', 25, 0.5, True, 5, np.float64(0.8176636380750716)),
 ('pearson', 25, 0.5, True, 10, np.float64(0.817881875788543)),
 ('pearson', 25, 0.5, True, 25, 

# Evaluación de similitud item-item

In [159]:
R_items = R_train.T

N = R_items.shape[0]

counts = np.diff(R_items.indptr)
sums = np.add.reduceat(R_items.data, R_items.indptr[:-1])
item_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts!=0)

### Función de cálculo de rating - Item-Item

Se crea una función capaz de implementar la predicción de ratings mediante **colaboración basada en ítems (item-item collaborative filtering)**. Esta función calcula la predicción del rating de un usuario para un ítem específico utilizando los ratings del usuario para ítems similares.

#### `predict_rating_item()`

La función toma como entrada los parámetros necesarios para realizar la predicción:

- **R**: Matriz dispersa (usuario × ítem) que contiene los ratings de entrenamiento
- **userId**: Índice del usuario para el cual se realiza la predicción
- **movieId**: Índice del ítem a predecir
- **neighbours**: Array con los índices de los ítems más similares
- **sims**: Array con las similitudes correspondientes a cada ítem vecino
- **K**: Número máximo de vecinos considerados
- **sim_threshold**: Valor mínimo de similitud para incluir un vecino en la predicción
- **means**: Promedio de ratings para cada ítem
- **weighting**: Booleano para aplicar ponderación de McLaughlin (opcional)
- **B**: Matriz de co-rated (elementos en común precomputados) (requerida si weighting=True)
- **beta**: Parámetro de escala para la ponderación de McLaughlin (por defecto 50)

#### Proceso de cálculo

1. **Extracción de ratings**: Se obtienen los ratings que el usuario ha dado a los ítems similares
2. **Filtrado**: Se descartan ítems que el usuario no calificó o cuya similitud es inferior al threshold
3. **Centrado**: Se restan los promedios de cada ítem vecino para normalizar las escalas
4. **Ponderación (opcional)**: Se aplica el método de McLaughlin para penalizar similitudes basadas en pocos datos comunes
5. **Combinación ponderada**: Se calcula la predicción como el promedio del ítem más una desviación ponderada

#### Fórmula matemática

$$\hat{r}_{u,i} = \bar{r}_i + \frac{\sum_{j \in N(i)} w_j \cdot \text{sim}(i,j) \cdot (r_{u,j} - \bar{r}_j)}{\sum_{j \in N(i)} |w_j \cdot \text{sim}(i,j)|}$$

Donde $w_j = \min(\text{corated}(i,j) / \beta, 1.0)$ cuando se aplica ponderación de McLaughlin.

Esta implementación permite hacer predicciones basadas en la similitud entre ítems, capturando patrones de preferencia a través de ítems relacionados.

In [160]:
def predict_rating_item(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    # --- get ratings for the movie ---
    ratings = R[userId, neighbours].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = B[movieId]

        weights = np.minimum(intersections / beta, 1.0)

        weights = weights[mask]

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[movieId] + numerator / denominator

    return pred


### Función de evaluación RMSE - Item-Item

Esta función calcula el error cuadrático medio (RMSE) de las predicciones generadas por el modelo de filtrado colaborativo basado en ítems. Permite comparar el desempeño del sistema bajo diferentes configuraciones de vecinos, similitud y ponderación.

#### `evaluate_rmse_item()`

**Parámetros:**

- **R**: Matriz dispersa de ratings de entrenamiento

- **test_df**: DataFrame de prueba con índices de usuario e ítem

- **neighbors_list**: Matriz de vecinos para cada ítem

- **sims_list**: Matriz de similitudes para cada ítem

- **K**: Número de vecinos considerados

- **sim_threshold**: Umbral mínimo de similitud

- **means**: Promedio de ratings por ítem

- **weighting**: Si aplica ponderación de McLaughlin

- **B**: Matriz de co-rated (elementos en común precomputados)

- **beta**: Parámetro de escala para ponderación


#### Proceso de cálculo

1. Para cada usuario e ítem en el set de prueba, predice el rating usando la función de predicción item-item.
2. Guarda la predicción y el rating real.
3. Calcula el RMSE entre las predicciones y los ratings reales.

#### Fórmula matemática

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (\hat{r}_i - r_i)^2}$$

Donde $\hat{r}_i$ es la predicción y $r_i$ el rating real.

Esta función permite evaluar la calidad de las predicciones basadas en similitud de ítems y comparar configuraciones del modelo.

In [161]:
def evaluate_rmse_item(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    preds = []
    truths = []

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating_item(
            R,
            user_idx,
            movie_idx,
            neighbors_list[movie_idx],
            sims_list[movie_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B,
            beta=beta
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse

## Submuestra de test (n=5.000)

In [162]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

In [163]:
for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    print(f"Dimensiones neighbours: {top_k_indices.shape}")
    print(f"Número de items en R_items: {R_items.shape[0]}")
    print(f"Longitud de item_means: {len(item_means)}")

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):
            weight=False
            rmse = evaluate_rmse_item(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                item_means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse))
            print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
Dimensiones neighbours: (25865, 100)
Número de items en R_items: 25865
Longitud de item_means: 138493


('cosine', 25, 0, False, np.float64(1.0056557408107982))


('cosine', 25, 0.3, False, np.float64(1.0183418489961007))


('cosine', 25, 0.5, False, np.float64(1.02821972394549))


...

('jaccard', 50, 0.5, False, np.float64(0.6839430922354743))


('jaccard', 100, 0, False, np.float64(0.9695694441791524))


('jaccard', 100, 0.3, False, np.float64(0.9774618390458099))




Similarity metric: 100%|██████████| 3/3 [00:10<00:00,  3.40s/it]

('jaccard', 100, 0.5, False, np.float64(0.6839430922354743))


In [164]:
results

[('cosine', 25, 0, False, np.float64(1.0056557408107982)),
 ('cosine', 25, 0.3, False, np.float64(1.0183418489961007)),
 ('cosine', 25, 0.5, False, np.float64(1.02821972394549)),
 ('cosine', 50, 0, False, np.float64(0.9839930175791636)),
 ('cosine', 50, 0.3, False, np.float64(1.0079954229036159)),
 ('cosine', 50, 0.5, False, np.float64(1.02821972394549)),
 ('cosine', 100, 0, False, np.float64(0.9819672028860388)),
 ('cosine', 100, 0.3, False, np.float64(1.0074222771151142)),
 ('cosine', 100, 0.5, False, np.float64(1.02821972394549)),
 ('pearson', 25, 0, False, np.float64(1.0053548899690898)),
 ('pearson', 25, 0.3, False, np.float64(1.0186697307054762)),
 ('pearson', 25, 0.5, False, np.float64(1.0282214619871746)),
 ('pearson', 50, 0, False, np.float64(0.9839066705010061)),
 ('pearson', 50, 0.3, False, np.float64(1.008054450397888)),
 ('pearson', 50, 0.5, False, np.float64(1.0282214619871746)),
 ('pearson', 100, 0, False, np.float64(0.981965511789455)),
 ('pearson', 100, 0.3, False, np.

## Submuestra de test (n=250.000)

In [165]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(250_000, random_state=42)

In [166]:
for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):
            weigth=False
            rmse = evaluate_rmse_item(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse))
            print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0


('cosine', 25, 0, False, np.float64(1.0031778877693514))


('cosine', 25, 0.3, False, np.float64(1.0010924383641229))


('cosine', 25, 0.5, False, np.float64(1.0136319308291375))


...

('jaccard', 50, 0.5, False, np.float64(0.6861883782523512))


('jaccard', 100, 0, False, np.float64(0.9782261085995155))


('jaccard', 100, 0.3, False, np.float64(0.9928851352947275))




Similarity metric: 100%|██████████| 3/3 [07:50<00:00, 156.69s/it]

('jaccard', 100, 0.5, False, np.float64(0.684946246016153))


In [167]:
results

[('cosine', 25, 0, False, np.float64(1.0031778877693514)),
 ('cosine', 25, 0.3, False, np.float64(1.0010924383641229)),
 ('cosine', 25, 0.5, False, np.float64(1.0136319308291375)),
 ('cosine', 50, 0, False, np.float64(0.9913165686950676)),
 ('cosine', 50, 0.3, False, np.float64(0.9937996966013667)),
 ('cosine', 50, 0.5, False, np.float64(1.0138803404843015)),
 ('cosine', 100, 0, False, np.float64(0.9891763670354068)),
 ('cosine', 100, 0.3, False, np.float64(0.9926210521453267)),
 ('cosine', 100, 0.5, False, np.float64(1.014133517536809)),
 ('pearson', 25, 0, False, np.float64(1.0032048512410878)),
 ('pearson', 25, 0.3, False, np.float64(1.0009137759770792)),
 ('pearson', 25, 0.5, False, np.float64(1.0136494470582744)),
 ('pearson', 50, 0, False, np.float64(0.9914112606246815)),
 ('pearson', 50, 0.3, False, np.float64(0.9936781333884452)),
 ('pearson', 50, 0.5, False, np.float64(1.0138978360523774)),
 ('pearson', 100, 0, False, np.float64(0.9891763895918905)),
 ('pearson', 100, 0.3, Fal

## Implementación de ponderación de McLaughlin

Se calculará McNally para el mejor tipo de similitud en el modelo item-item, que en este caso es cosine, para esta se iterará para 3 Valores de k vecinos, 3 valores de umbral de similitud, y 5 posibles valores de ponderación de McLaughlin (incluyendo NO implementarla)

## Submuestra de test (n=5.000)

In [179]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

In [180]:
for metric in tqdm(["cosine"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/items_{metric}_top100_co_rated.npz") as data:
        ttop_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.3, 0.5], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse = evaluate_rmse_item(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse))
                    print((metric, K, sim_threshold, weight, None, rmse))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse = evaluate_rmse_item(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse))
                        print((metric, K, sim_threshold, weight, beta, rmse))

    

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
min co_rated: 0
max co_rated: 3642


('cosine', 25, 0, False, None, np.float64(1.0056557408107982))
('cosine', 25, 0, True, 5, np.float64(1.0126385777047253))
('cosine', 25, 0, True, 10, np.float64(1.0157086927550556))
('cosine', 25, 0, True, 25, np.float64(1.0193661703168475))
('cosine', 25, 0, True, 50, np.float64(1.0214319011596535))


('cosine', 25, 0, True, 100, np.float64(1.0221340048526006))
('cosine', 25, 0.3, False, None, np.float64(1.0183418489961007))
('cosine', 25, 0.3, True, 5, np.float64(1.0224074880223366))
('cosine', 25, 0.3, True, 10, np.float64(1.0241219892773166))
('cosine', 25, 0.3, True, 25, np.float64(1.0266513689711505))
('cosine', 25, 0.3, True, 50, np.float64(1.0282688268408187))


('cosine', 25, 0.3, True, 100, np.float64(1.028275347519443))
('cosine', 25, 0.5, False, None, np.float64(1.02821972394549))
('cosine', 25, 0.5, True, 5, np.float64(1.0314686233354695))
('cosine', 25, 0.5, True, 10, np.float64(1.0360754753213246))
('cosine', 25, 0.5, True, 25, np.float64(1.0411654793673404))
('cosine', 25, 0.5, True, 50, np.float64(1.0412495018466568))


...

('cosine', 100, 0, False, None, np.float64(0.9819672028860388))
('cosine', 100, 0, True, 5, np.float64(0.9850647099463227))
('cosine', 100, 0, True, 10, np.float64(0.9875792172572956))
('cosine', 100, 0, True, 25, np.float64(0.9901409535083007))
('cosine', 100, 0, True, 50, np.float64(0.9919377749551777))


('cosine', 100, 0, True, 100, np.float64(0.9928444176866846))
('cosine', 100, 0.3, False, None, np.float64(1.0074222771151142))
('cosine', 100, 0.3, True, 5, np.float64(1.0094818743883287))
('cosine', 100, 0.3, True, 10, np.float64(1.0118338443134995))
('cosine', 100, 0.3, True, 25, np.float64(1.0145083368974257))
('cosine', 100, 0.3, True, 50, np.float64(1.0154976250212144))


('cosine', 100, 0.3, True, 100, np.float64(1.0153853159046924))
('cosine', 100, 0.5, False, None, np.float64(1.02821972394549))
('cosine', 100, 0.5, True, 5, np.float64(1.0314686233354695))
('cosine', 100, 0.5, True, 10, np.float64(1.0360754753213246))
('cosine', 100, 0.5, True, 25, np.float64(1.0411654793673404))
('cosine', 100, 0.5, True, 50, np.float64(1.0412495018466568))




Similarity metric: 100%|██████████| 1/1 [00:21<00:00, 21.86s/it]

('cosine', 100, 0.5, True, 100, np.float64(1.0414209747277385))


In [181]:
results

[('cosine', 25, 0, False, None, np.float64(1.0056557408107982)),
 ('cosine', 25, 0, True, 5, np.float64(1.0126385777047253)),
 ('cosine', 25, 0, True, 10, np.float64(1.0157086927550556)),
 ('cosine', 25, 0, True, 25, np.float64(1.0193661703168475)),
 ('cosine', 25, 0, True, 50, np.float64(1.0214319011596535)),
 ('cosine', 25, 0, True, 100, np.float64(1.0221340048526006)),
 ('cosine', 25, 0.3, False, None, np.float64(1.0183418489961007)),
 ('cosine', 25, 0.3, True, 5, np.float64(1.0224074880223366)),
 ('cosine', 25, 0.3, True, 10, np.float64(1.0241219892773166)),
 ('cosine', 25, 0.3, True, 25, np.float64(1.0266513689711505)),
 ('cosine', 25, 0.3, True, 50, np.float64(1.0282688268408187)),
 ('cosine', 25, 0.3, True, 100, np.float64(1.028275347519443)),
 ('cosine', 25, 0.5, False, None, np.float64(1.02821972394549)),
 ('cosine', 25, 0.5, True, 5, np.float64(1.0314686233354695)),
 ('cosine', 25, 0.5, True, 10, np.float64(1.0360754753213246)),
 ('cosine', 25, 0.5, True, 25, np.float64(1.041

## Submuestra de test (n=250.000)

In [ ]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

In [ ]:
for metric in tqdm(["cosine"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/items_{metric}_top100_co_rated.npz") as data:
        ttop_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.3, 0.5], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse = evaluate_rmse_item(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse))
                    print((metric, K, sim_threshold, weight, None, rmse))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse = evaluate_rmse_item(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse))
                        print((metric, K, sim_threshold, weight, beta, rmse))

    

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
min co_rated: 0
max co_rated: 3642


('cosine', 25, 0, False, None, np.float64(1.0031778877693514))
('cosine', 25, 0, True, 5, np.float64(1.0072107972053828))
('cosine', 25, 0, True, 10, np.float64(1.0102499598554469))
('cosine', 25, 0, True, 25, np.float64(1.0141272466170776))
('cosine', 25, 0, True, 50, np.float64(1.016337502623393))


('cosine', 25, 0, True, 100, np.float64(1.0175397008774734))
('cosine', 25, 0.3, False, None, np.float64(1.0010924383641229))
('cosine', 25, 0.3, True, 5, np.float64(1.005123795875243))
('cosine', 25, 0.3, True, 10, np.float64(1.007825211261411))
('cosine', 25, 0.3, True, 25, np.float64(1.011419857939181))
('cosine', 25, 0.3, True, 50, np.float64(1.0135760897314654))


('cosine', 25, 0.3, True, 100, np.float64(1.0145473245231709))
('cosine', 25, 0.5, False, None, np.float64(1.0136319308291375))
('cosine', 25, 0.5, True, 5, np.float64(1.0204251048744382))
('cosine', 25, 0.5, True, 10, np.float64(1.0241174462804528))
('cosine', 25, 0.5, True, 25, np.float64(1.0272770185999436))
('cosine', 25, 0.5, True, 50, np.float64(1.0286480969533955))


...

('cosine', 100, 0, False, None, np.float64(0.9891763670354068))
('cosine', 100, 0, True, 5, np.float64(0.9911203842963496))
('cosine', 100, 0, True, 10, np.float64(0.9927350429300845))
('cosine', 100, 0, True, 25, np.float64(0.9952292406919283))
('cosine', 100, 0, True, 50, np.float64(0.9966000409085767))


('cosine', 100, 0, True, 100, np.float64(0.9972327514672497))
('cosine', 100, 0.3, False, None, np.float64(0.9926210521453267))
('cosine', 100, 0.3, True, 5, np.float64(0.9954419422195828))
('cosine', 100, 0.3, True, 10, np.float64(0.9974461284574168))
('cosine', 100, 0.3, True, 25, np.float64(1.000396601518433))
('cosine', 100, 0.3, True, 50, np.float64(1.0021288514818185))


('cosine', 100, 0.3, True, 100, np.float64(1.0028629048386708))
('cosine', 100, 0.5, False, None, np.float64(1.014133517536809))
('cosine', 100, 0.5, True, 5, np.float64(1.0209285260016305))
('cosine', 100, 0.5, True, 10, np.float64(1.024629229532844))
('cosine', 100, 0.5, True, 25, np.float64(1.027806757483168))
('cosine', 100, 0.5, True, 50, np.float64(1.0291812505537132))




Similarity metric: 100%|██████████| 1/1 [17:07<00:00, 1027.50s/it]

('cosine', 100, 0.5, True, 100, np.float64(1.0306602958815112))


In [185]:
results

[('cosine', 25, 0, False, None, np.float64(1.0031778877693514)),
 ('cosine', 25, 0, True, 5, np.float64(1.0072107972053828)),
 ('cosine', 25, 0, True, 10, np.float64(1.0102499598554469)),
 ('cosine', 25, 0, True, 25, np.float64(1.0141272466170776)),
 ('cosine', 25, 0, True, 50, np.float64(1.016337502623393)),
 ('cosine', 25, 0, True, 100, np.float64(1.0175397008774734)),
 ('cosine', 25, 0.3, False, None, np.float64(1.0010924383641229)),
 ('cosine', 25, 0.3, True, 5, np.float64(1.005123795875243)),
 ('cosine', 25, 0.3, True, 10, np.float64(1.007825211261411)),
 ('cosine', 25, 0.3, True, 25, np.float64(1.011419857939181)),
 ('cosine', 25, 0.3, True, 50, np.float64(1.0135760897314654)),
 ('cosine', 25, 0.3, True, 100, np.float64(1.0145473245231709)),
 ('cosine', 25, 0.5, False, None, np.float64(1.0136319308291375)),
 ('cosine', 25, 0.5, True, 5, np.float64(1.0204251048744382)),
 ('cosine', 25, 0.5, True, 10, np.float64(1.0241174462804528)),
 ('cosine', 25, 0.5, True, 25, np.float64(1.0272